# Exploratory Data Analysis of U.S. Flight Operations (2020–2025)

## Objective

This notebook performs a concise exploratory data analysis of the flight dataset stored in the Amazon S3 Silver layer using PySpark on Amazon EMR.

The analysis focuses on 29 business-relevant columns covering:

- Time dimensions
- Airlines, airports and routes
- Departure and arrival delays
- Delay causes
- Cancellations and diversions
- Flight distance and operational duration

The dataset is processed using Spark DataFrames so that the analysis remains distributed and suitable for the approximately 40.9 million records available in the source.

In [1]:
import pyspark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
4,application_1784518578866_0005,pyspark,idle,Link,Link,✔


SparkSession available as 'spark'.


## 1. Load the Silver-Layer Dataset

The partitioned Parquet dataset is loaded directly from Amazon S3. Parquet is appropriate for this analysis because it supports column pruning and distributed processing in Spark.

In [2]:
silver_df = spark.read.parquet(
    "s3a://airline-dataset-2020-2025/Silver/Flight_Data_2020_2025/"
)

print("Rows :", silver_df.count())
print("Columns :", len(silver_df.columns))

('Rows :', 40910253)
('Columns :', 120)

## 2. Review Available Columns

The source dataset contains 120 columns. Listing them helps verify the schema before selecting only the variables required for this EDA.

In [3]:
print(silver_df.columns)

['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Marketing_Airline_Network', 'Operated_or_Branded_Code_Share_Partners', 'DOT_ID_Marketing_Airline', 'IATA_Code_Marketing_Airline', 'Flight_Number_Marketing_Airline', 'Originally_Scheduled_Code_Share_Airline', 'DOT_ID_Originally_Scheduled_Code_Share_Airline', 'IATA_Code_Originally_Scheduled_Code_Share_Airline', 'Flight_Num_Originally_Scheduled_Code_Share_Airline', 'Operating_Airline', 'DOT_ID_Operating_Airline', 'IATA_Code_Operating_Airline', 'Tail_Number', 'Flight_Number_Operating_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'Tax

## 3. Select the EDA Columns

Only the 29 approved columns are retained. Reducing the width of the DataFrame lowers unnecessary I/O and memory consumption on the EMR cluster.

In [4]:
from pyspark.sql import functions as F
selected_columns = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek", "FlightDate",
    "Marketing_Airline_Network", "Origin", "OriginState", "Dest", "DestState",
    "CRSDepTime", "CRSArrTime", "DepDelay", "ArrDelay", "DepDel15", "ArrDel15",
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay",
    "LateAircraftDelay", "Cancelled", "Diverted", "Distance", "AirTime",
    "TaxiOut", "TaxiIn", "Flight_Number_Marketing_Airline"
]

eda_df = silver_df.select(*selected_columns)
# Ensure FlightDate has the correct datatype
eda_df = eda_df.withColumn(
    "FlightDate",
    F.to_date("FlightDate")
)


eda_df.printSchema()


root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- DepDel15: double (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)
 |-- Cancelled: double (nullable = true)
 |-- Diverted: double (nullable

## Optional DataFrame Persistence

Persistence may improve performance when the same reduced DataFrame is scanned repeatedly. Because the cluster has limited capacity, it should be enabled only when repeated actions justify the memory usage and released with `unpersist()` after the analysis.

In [ ]:
# eda_df = eda_df.persist()

# # Run EDA operations

# eda_df.unpersist()

## 4. Basic Dataset Inspection

This step confirms the number of selected columns, total record count, data types and a small sample of records.

**Observed size:** 40,910,253 rows and 29 selected columns.

In [5]:
print("Number of columns:", len(eda_df.columns))
print("Number of rows:", eda_df.count())

eda_df.printSchema()
eda_df.show(5, truncate=False)

('Number of columns:', 29)
('Number of rows:', 40910253)
root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- DepDel15: double (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-- CarrierDelay: double (nullable = true)
 |-- WeatherDelay: double (nullable = true)
 |-- NASDelay: double (nullable = true)
 |-- SecurityDelay: double (nullable = true)
 |-- LateAircraftDelay: double (nullable = true)
 |-- Cancelled:

# Correct important data types

The date and calendar fields are explicitly converted to suitable Spark data types. Correct typing is required for date filtering, grouping and future time-series analysis.

In [6]:
eda_df = (
    eda_df
    .withColumn("FlightDate", F.to_date("FlightDate"))
    .withColumn("Year", F.col("Year").cast("int"))
    .withColumn("Quarter", F.col("Quarter").cast("int"))
    .withColumn("Month", F.col("Month").cast("int"))
    .withColumn("DayofMonth", F.col("DayofMonth").cast("int"))
    .withColumn("DayOfWeek", F.col("DayOfWeek").cast("int"))
)

## 5. Missing-Value Analysis

Null counts are calculated for every selected column. Operational fields such as arrival delay, air time and taxi time may be missing for cancelled or otherwise incomplete flight records.

In [7]:
row_count = eda_df.count()

null_summary = eda_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in eda_df.columns
])

null_summary.show(truncate=False)

+----+-------+-----+----------+---------+----------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+---------+--------+--------+-------+-------+------+-------------------------------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Marketing_Airline_Network|Origin|OriginState|Dest|DestState|CRSDepTime|CRSArrTime|DepDelay|ArrDelay|DepDel15|ArrDel15|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|Cancelled|Diverted|Distance|AirTime|TaxiOut|TaxiIn|Flight_Number_Marketing_Airline|
+----+-------+-----+----------+---------+----------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+---------+--------+--------+-------+-------+------+-------------------------------+
|0   |0      |0    |0         |0     

### Missing-Value Percentages

Percentages make it easier to compare missingness across columns.

Key observations from the output:

- Core identifiers, dates, airports, schedules, distance, cancellation and diversion fields have no material missingness.
- `DepDelay` and `DepDel15` are missing in approximately **2.19%** of records.
- `ArrDelay`, `ArrDel15` and `AirTime` are missing in approximately **2.48%** of records.
- `TaxiOut` and `TaxiIn` are missing in approximately **2.23%** and **2.26%** of records.
- Delay-cause fields are null in approximately **81.31%** of records because these values are generally populated only when a qualifying delay occurs.

In [8]:
null_percentage = eda_df.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) / F.lit(row_count) * 100,
        2
    ).alias(c)
    for c in eda_df.columns
])

null_percentage.show(truncate=False)

+----+-------+-----+----------+---------+----------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+---------+--------+--------+-------+-------+------+-------------------------------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Marketing_Airline_Network|Origin|OriginState|Dest|DestState|CRSDepTime|CRSArrTime|DepDelay|ArrDelay|DepDel15|ArrDel15|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|Cancelled|Diverted|Distance|AirTime|TaxiOut|TaxiIn|Flight_Number_Marketing_Airline|
+----+-------+-----+----------+---------+----------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+---------+--------+--------+-------+-------+------+-------------------------------+
|0.0 |0.0    |0.0  |0.0       |0.0   

### Treat Null Delay Causes as Zero

For aggregate delay-cause analysis, null delay-cause values are replaced with zero. This means that a missing cause is treated as no recorded minutes for that cause.

This transformation is suitable for descriptive aggregation, but the original null pattern should be retained or reconsidered for predictive modelling.

In [9]:
delay_columns = [
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay"
]

eda_df = eda_df.fillna(0, subset=delay_columns)

## 6. Duplicate Flight Check

A composite business key is used to identify repeated scheduled-flight records:

`FlightDate + Airline + Flight Number + Origin + Destination + Scheduled Departure Time`

The displayed result contains no duplicate records under this definition.

In [10]:
duplicate_flights = (
    eda_df
    .groupBy(
        "FlightDate",
        "Marketing_Airline_Network",
        "Flight_Number_Marketing_Airline",
        "Origin",
        "Dest",
        "CRSDepTime"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_flights.show(20, truncate=False)

+----------+-------------------------+-------------------------------+------+----+----------+-----+
|FlightDate|Marketing_Airline_Network|Flight_Number_Marketing_Airline|Origin|Dest|CRSDepTime|count|
+----------+-------------------------+-------------------------------+------+----+----------+-----+
+----------+-------------------------+-------------------------------+------+----+----------+-----+

## 7. Numerical Summary

Descriptive statistics are calculated for delay, distance, air-time and taxi-time fields.

Important observations:

- Median departure delay is **−2 minutes**, indicating that at least half of recorded flights departed on time or early.
- Median arrival delay is **−7 minutes**, indicating that at least half arrived early relative to schedule.
- Mean departure and arrival delays are positive because a smaller number of extreme delays pull the averages upward.
- Maximum delays above 7,000 minutes indicate severe outliers that should be handled carefully in modelling and visualisation.
- The typical flight covers about **642 miles** at the median and has approximately **93 minutes** of air time.

In [11]:
numerical_columns = [
    "DepDelay", "ArrDelay",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay", 
    "Distance", "AirTime", "TaxiOut", "TaxiIn"
]

eda_df.select(numerical_columns).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show(truncate=False)

+-------+------------------+-----------------+------------------+------------------+------------------+--------------------+-----------------+-----------------+------------------+-----------------+-----------------+
|summary|DepDelay          |ArrDelay         |CarrierDelay      |WeatherDelay      |NASDelay          |SecurityDelay       |LateAircraftDelay|Distance         |AirTime           |TaxiOut          |TaxiIn           |
+-------+------------------+-----------------+------------------+------------------+------------------+--------------------+-----------------+-----------------+------------------+-----------------+-----------------+
|count  |40013735          |39895374         |40910253          |40910253          |40910253          |40910253            |40910253         |40910253         |39895374          |39997567         |39983837         |
|mean   |10.937542346396805|5.242046383623324|4.720685374397465 |0.7995908996211781|2.4566242843817174|0.025418640163383983|5.0684305716

## 8. Yearly Flight and Reliability Trends

The yearly summary evaluates traffic volume, average delays, 15-minute delay rates, cancellation rates and diversion rates.

Main pattern:

- Flight volume increased from about **5.02 million in 2020** to **7.74 million in 2025**.
- Average arrival delay rose from **−4.86 minutes in 2020** to **8.53 minutes in 2025**.
- The arrival-delay rate increased to **22.18% in 2025**, the highest value in the displayed period.
- The exceptional **5.99% cancellation rate in 2020** reflects abnormal operating conditions; later years are substantially lower.
- Diversions remain rare, below **0.30%** in every year.

In [12]:
yearly_summary = (
    eda_df
    .groupBy("Year")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.avg("DepDel15") * 100, 2).alias("DepDelayRatePct"),
        F.round(F.avg("ArrDel15") * 100, 2).alias("ArrDelayRatePct"),
        F.round(F.avg("Cancelled") * 100, 2).alias("CancellationRatePct"),
        F.round(F.avg("Diverted") * 100, 2).alias("DiversionRatePct")
    )
    .orderBy("Year")
)

yearly_summary.show()

+----+------------+-----------+-----------+---------------+---------------+-------------------+----------------+
|Year|TotalFlights|AvgDepDelay|AvgArrDelay|DepDelayRatePct|ArrDelayRatePct|CancellationRatePct|DiversionRatePct|
+----+------------+-----------+-----------+---------------+---------------+-------------------+----------------+
|2020|     5022397|       2.06|      -4.86|           9.17|           9.95|               5.99|            0.17|
|2021|     6311871|       9.47|       3.29|          17.32|          17.27|               1.76|            0.24|
|2022|     7013508|      12.48|       6.96|          21.13|          20.95|               2.71|            0.24|
|2023|     7278739|      12.21|       6.63|           20.3|          20.43|               1.29|            0.24|
|2024|     7546968|      12.51|        7.0|          20.35|          20.62|               1.36|            0.25|
|2025|     7736770|      13.52|       8.53|          21.52|          22.18|               1.53| 

## 9. Monthly Seasonality

Monthly aggregation reveals seasonal differences in operational performance.

- **June and July** have the highest average delays and delayed-flight percentages.
- July records the highest average arrival delay at approximately **12.75 minutes** and the highest delayed-flight rate at **25.11%**.
- **September through November** show the strongest performance, with average arrival delays close to one minute and delayed-flight rates near 15%.
- December performance weakens again, consistent with holiday demand and winter disruption risk.

In [13]:
monthly_summary = (
    eda_df
    .groupBy("Month")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.avg("ArrDel15") * 100, 2).alias("DelayedFlightsPct")
    )
    .orderBy("Month")
)

monthly_summary.show()

+-----+------------+-----------+-----------+-----------------+
|Month|TotalFlights|AvgDepDelay|AvgArrDelay|DelayedFlightsPct|
+-----+------------+-----------+-----------+-----------------+
|    1|     3358992|      10.15|       3.77|            18.46|
|    2|     3141722|       9.09|       2.71|            17.63|
|    3|     3668894|       9.16|       3.17|            17.89|
|    4|     3246144|       9.73|       4.06|            18.19|
|    5|     3249065|      11.72|       6.36|            19.82|
|    6|     3352655|      16.12|      11.32|            24.23|
|    7|     3617203|       17.4|      12.75|            25.11|
|    8|     3590730|      12.68|       7.49|            20.78|
|    9|     3341566|       7.12|       1.22|            15.15|
|   10|     3525386|       7.37|       1.47|            15.63|
|   11|     3378386|       7.22|       1.03|            15.47|
|   12|     3439510|      13.06|       7.06|            21.21|
+-----+------------+-----------+-----------+-----------

## 10. Quarterly Performance

Quarter-level analysis combines seasonality with year-specific operating conditions.

The most unusual result is **2020 Q2**, which has a cancellation rate of **19.73%** and sharply reduced flight volume. In the later displayed years, Q2 and Q3 frequently show higher arrival delays than Q1 and Q4.

In [14]:
quarterly_summary = (
    eda_df
    .groupBy("Year", "Quarter")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.avg("Cancelled") * 100, 2).alias("CancellationRatePct")
    )
    .orderBy("Year", "Quarter")
)

quarterly_summary.show()

+----+-------+------------+-----------+-------------------+
|Year|Quarter|TotalFlights|AvgArrDelay|CancellationRatePct|
+----+-------+------------+-----------+-------------------+
|2020|      1|     1984933|      -2.35|               6.69|
|2020|      2|      760914|     -10.65|              19.73|
|2020|      3|     1114623|      -6.11|               0.88|
|2020|      4|     1161927|      -4.65|               0.72|
|2021|      1|     1196680|      -3.43|               2.53|
|2021|      2|     1567774|        2.7|                0.9|
|2021|      3|     1795113|       7.69|               2.04|
|2021|      4|     1752304|       3.88|               1.71|
|2022|      1|     1674231|       5.41|               4.07|
|2022|      2|     1785297|       8.74|               2.46|
|2022|      3|     1812830|       6.93|               1.96|
|2022|      4|     1741150|       6.65|               2.44|
|2023|      1|     1726340|       7.15|               1.65|
|2023|      2|     1826883|       9.15| 

## 11. Day-of-Week Performance

The BTS coding used here is:

1 = Monday, 2 = Tuesday, 3 = Wednesday, 4 = Thursday,  
5 = Friday, 6 = Saturday, 7 = Sunday.

Key observations:

- **Sunday** has the highest average arrival delay at **7.38 minutes** and the highest arrival-delay rate at **21.04%**.
- **Friday** is similarly delay-prone.
- **Tuesday** performs best, with an average arrival delay of **2.18 minutes** and a delay rate of **16.39%**.
- Day of week is therefore a useful feature for later delay-prediction models.

In [15]:
day_summary = (
    eda_df
    .groupBy("DayOfWeek")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.avg("ArrDel15") * 100, 2).alias("ArrDelayRatePct")
    )
    .orderBy("DayOfWeek")
)

day_summary.show()

+---------+------------+-----------+-----------+---------------+
|DayOfWeek|TotalFlights|AvgDepDelay|AvgArrDelay|ArrDelayRatePct|
+---------+------------+-----------+-----------+---------------+
|        1|     6105492|      11.46|       5.74|          19.43|
|        2|     5583713|       8.43|       2.18|          16.39|
|        3|     5686712|       8.73|       2.94|          17.12|
|        4|     6069233|      11.32|       6.24|          20.13|
|        5|     6105950|      12.29|       7.08|          20.97|
|        6|     5327980|      11.02|       4.69|          18.64|
|        7|     6031173|       13.0|       7.38|          21.04|
+---------+------------+-----------+-----------+---------------+

## 12. Airline Performance

This comparison evaluates flight volume and reliability by marketing airline.

- **American Airlines (AA)** has the largest number of flights in the dataset.
- Among the highest-volume carriers, **Delta (DL)** shows the strongest reliability, with an average arrival delay of **1.59 minutes** and a 15.03% arrival-delay rate.
- **JetBlue (B6)** and **Frontier (F9)** have relatively high average arrival delays and delay rates.
- **Allegiant (G4)** has the highest displayed cancellation rate at **3.69%**.

These comparisons are descriptive and do not control for route mix, weather exposure, airport congestion or flight distance.

In [16]:
airline_summary = (
    eda_df
    .groupBy("Marketing_Airline_Network")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.avg("ArrDel15") * 100, 2).alias("ArrDelayRatePct"),
        F.round(F.avg("Cancelled") * 100, 2).alias("CancellationRatePct")
    )
    .orderBy(F.desc("TotalFlights"))
)

airline_summary.show(20)

+-------------------------+------------+-----------+-----------+---------------+-------------------+
|Marketing_Airline_Network|TotalFlights|AvgDepDelay|AvgArrDelay|ArrDelayRatePct|CancellationRatePct|
+-------------------------+------------+-----------+-----------+---------------+-------------------+
|                       AA|    10407897|      12.05|       7.14|           19.9|                2.5|
|                       DL|     8526129|       8.32|       1.59|          15.03|               1.63|
|                       WN|     7582834|      11.35|        4.2|          20.14|                2.4|
|                       UA|     7427635|      10.99|       5.28|          18.65|               2.41|
|                       AS|     2238407|       5.33|       2.39|          18.02|               1.76|
|                       B6|     1366470|       18.1|      12.15|          26.97|               2.55|
|                       NK|     1278352|      13.78|       8.25|          22.95|           

## 13. Origin-Airport Analysis

The origin-airport summary measures departure volume, departure delay and taxi-out time.

- **Atlanta (ATL)** is the busiest displayed origin airport.
- **Dallas/Fort Worth (DFW)** and **Orlando (MCO)** have relatively high average departure delays among the busiest airports.
- **JFK, EWR, LGA and ORD** show high taxi-out times, suggesting greater surface congestion or operational complexity.
- Airport comparisons should be interpreted alongside traffic volume, runway configuration, weather and route mix.

In [17]:
origin_summary = (
    eda_df
    .groupBy("Origin", "OriginState")
    .agg(
        F.count("*").alias("TotalDepartures"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.avg("TaxiOut"), 2).alias("AvgTaxiOut")
    )
    .orderBy(F.desc("TotalDepartures"))
)

origin_summary.show(20)

+------+-----------+---------------+-----------+----------+
|Origin|OriginState|TotalDepartures|AvgDepDelay|AvgTaxiOut|
+------+-----------+---------------+-----------+----------+
|   ATL|         GA|        1915120|       9.69|     15.86|
|   ORD|         IL|        1778804|      12.56|     22.77|
|   DFW|         TX|        1705528|      14.96|     19.31|
|   DEN|         CO|        1683092|      13.92|      17.9|
|   CLT|         NC|        1347556|      10.67|     21.74|
|   LAX|         CA|        1086362|       9.14|     16.99|
|   SEA|         WA|        1026212|        7.3|     19.54|
|   PHX|         AZ|        1021572|      10.16|     15.32|
|   LAS|         NV|         999915|      12.42|     17.11|
|   IAH|         TX|         904228|      12.91|     19.07|
|   MCO|         FL|         859437|      15.26|     17.72|
|   LGA|         NY|         824195|      11.51|     23.44|
|   DTW|         MI|         771724|       9.21|     17.18|
|   EWR|         NJ|         754423|    

## 14. Route Analysis

Routes are ranked by total flight count and summarised using distance, air time and arrival delay.

- **LAX–SFO** and **SFO–LAX** are the most frequent displayed directional routes.
- Short-haul routes dominate the highest-frequency list.
- Opposite directions on the same airport pair can have different air times and delays because of winds, airspace flow and scheduling conditions.
- Route-level metrics can support dashboard filtering and later route-specific predictive features.

In [18]:
route_summary = (
    eda_df
    .groupBy("Origin", "Dest")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("Distance"), 2).alias("AvgDistance"),
        F.round(F.avg("AirTime"), 2).alias("AvgAirTime"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay")
    )
    .orderBy(F.desc("TotalFlights"))
)

route_summary.show(20)

+------+----+------------+-----------+----------+-----------+
|Origin|Dest|TotalFlights|AvgDistance|AvgAirTime|AvgArrDelay|
+------+----+------------+-----------+----------+-----------+
|   LAX| SFO|       65140|      337.0|     55.65|       4.49|
|   SFO| LAX|       65091|      337.0|     56.05|       1.65|
|   HNL| OGG|       60102|     100.09|     22.08|       2.72|
|   OGG| HNL|       60101|     100.09|     24.62|       2.79|
|   LAX| LAS|       57531|      236.0|     44.05|       6.84|
|   LAS| LAX|       57433|      236.0|     42.83|       6.36|
|   ORD| LGA|       56551|      733.0|     98.15|       8.94|
|   LGA| ORD|       56533|      733.0|    116.46|       4.67|
|   JFK| LAX|       53628|     2475.0|    328.16|      -1.08|
|   LAX| JFK|       53619|     2475.0|    288.74|       2.88|
|   PHX| DEN|       49907|      602.0|     81.18|       5.49|
|   DEN| PHX|       49859|      602.0|     88.96|       7.41|
|   BOS| DCA|       49205|      399.0|      75.4|       3.75|
|   DCA|

## 15. Delay-Cause Analysis

Total recorded delay minutes are summed by cause.

The leading contributors are:

1. **Late aircraft delay**
2. **Carrier delay**
3. **NAS delay**
4. **Weather delay**
5. **Security delay**

Late incoming aircraft is the largest recorded cause, indicating that disruption can propagate through an airline's network. These totals are influenced by both frequency and duration, so percentages or average minutes per delayed flight may be added in deeper analysis.

In [19]:
delay_causes = eda_df.agg(
    F.round(F.sum("CarrierDelay"), 2).alias("CarrierDelay"),
    F.round(F.sum("WeatherDelay"), 2).alias("WeatherDelay"),
    F.round(F.sum("NASDelay"), 2).alias("NASDelay"),
    F.round(F.sum("SecurityDelay"), 2).alias("SecurityDelay"),
    F.round(F.sum("LateAircraftDelay"), 2).alias("LateAircraftDelay")
)

delay_causes.show()

+------------+------------+------------+-------------+-----------------+
|CarrierDelay|WeatherDelay|    NASDelay|SecurityDelay|LateAircraftDelay|
+------------+------------+------------+-------------+-----------------+
|1.93124433E8| 3.2711466E7|1.00501121E8|    1039883.0|     2.07350777E8|
+------------+------------+------------+-------------+-----------------+

## 16. Data-Quality Validation

Basic validation checks are applied to distance, flight duration, taxi duration, cancellation flags and diversion flags.

The results show:

- No non-positive flight distances
- No negative air-time values
- No negative taxi-in or taxi-out values
- No invalid cancellation or diversion indicators

The selected columns therefore pass these basic logical checks.

In [20]:
eda_df.select(
    F.sum(F.when(F.col("Distance") <= 0, 1).otherwise(0))
        .alias("InvalidDistance"),

    F.sum(F.when(F.col("AirTime") < 0, 1).otherwise(0))
        .alias("NegativeAirTime"),

    F.sum(F.when(F.col("TaxiOut") < 0, 1).otherwise(0))
        .alias("NegativeTaxiOut"),

    F.sum(F.when(F.col("TaxiIn") < 0, 1).otherwise(0))
        .alias("NegativeTaxiIn"),

    F.sum(F.when(~F.col("Cancelled").isin(0, 1), 1).otherwise(0))
        .alias("InvalidCancelled"),

    F.sum(F.when(~F.col("Diverted").isin(0, 1), 1).otherwise(0))
        .alias("InvalidDiverted")
).show()

+---------------+---------------+---------------+--------------+----------------+---------------+
|InvalidDistance|NegativeAirTime|NegativeTaxiOut|NegativeTaxiIn|InvalidCancelled|InvalidDiverted|
+---------------+---------------+---------------+--------------+----------------+---------------+
|              0|              0|              0|             0|               0|              0|
+---------------+---------------+---------------+--------------+----------------+---------------+

# Final Conclusion

The PySpark EDA confirms that the Silver-layer flight dataset is large, structured and suitable for distributed analysis on Amazon EMR. The selected dataset contains **40,910,253 records across 29 analytical columns**, with no duplicates under the defined composite flight key and no basic logical errors in distance, duration, cancellation or diversion fields.

The main business findings are:

- Flight traffic recovered and expanded strongly after 2020, reaching approximately **7.74 million flights in 2025**.
- Operational reliability deteriorated across the period: average arrival delay and the proportion of flights delayed by at least 15 minutes were highest in 2025.
- Delay risk is seasonal. **June and July** are the weakest months, while **September through November** generally perform better.
- **Sunday and Friday** are the most delay-prone days, whereas **Tuesday** has the strongest displayed reliability.
- Airline performance differs materially. Delta performs strongly among major carriers, while JetBlue, Frontier and Allegiant show weaker results on selected delay or cancellation measures.
- Large hub airports handle the greatest traffic but several—particularly JFK, EWR, LGA and ORD—also show elevated taxi-out times.
- **Late aircraft delay** is the largest recorded delay cause, followed by carrier and NAS delays, showing that delay propagation and network congestion are major operational issues.
- Extreme delay values are present, so robust statistics, outlier treatment and suitable transformations should be considered before machine-learning model development.

Overall, the selected variables provide a strong foundation for airline and airport dashboards, reliability KPIs, route analysis and supervised models predicting `DepDel15`, `ArrDel15`, cancellations or arrival-delay duration.